# Mask erosion loss analysis
Reproducible notebook to generate reports in `analysis/reports`:
- `mask_erosion_loss_report.csv`
- `mask_erosion_boundary_white_report.csv`
- `mask_erosion_loss_flags_kernel3.csv`

## Metric definitions
- `kernel=0` means **no erosion** (baseline mask).
- `boundary` is the 1-pixel ring of the mask. We compute it as: `inner = erode(mask, 3x3, 1)` and `boundary = mask - inner`.
- `boundary_pixels` = number of pixels in that ring (`sum(boundary)`).
- `white_pixels` = image pixels where all R,G,B channels are >= `white_threshold`.
- `white_boundary_pixels` = white pixels that also belong to `boundary`.
- `white_boundary_pct` = `100 * white_boundary_pixels / boundary_pixels`.
- `white_boundary_pct_of_image` = `100 * white_boundary_pixels / image_pixels`.
- `image_pixels` comes from image size: `height * width`.

Important interpretation:
- `white_boundary_pct` is **not** percent of the full image.
- It is only percent of the mask boundary ring.

In [11]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# paths are relative to the analysis folder
masks_dir = Path('..') / 'data' / 'PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED'
reports_dir = Path('reports')
reports_dir.mkdir(parents=True, exist_ok=True)

loss_out = reports_dir / 'mask_erosion_loss_report_PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED.csv'
boundary_out = reports_dir / 'mask_erosion_boundary_white_report_PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED.csv'

kernel_sizes = [1, 2, 3, 5]
white_threshold = 245

mask_paths = sorted(masks_dir.glob('*_mask.png'))
print('masks_dir:', masks_dir.resolve())
print('reports_dir:', reports_dir.resolve())
print('mask files found:', len(mask_paths))

masks_dir: /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED
reports_dir: /home/daniduhnev/projects/master-thesis/analysis/reports
mask files found: 134


In [12]:
loss_rows = []

for mask_path in mask_paths:
    study_id = mask_path.name.replace('_mask.png', '')

    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_gray is None:
        continue

    mask_bin = (mask_gray > 0).astype(np.uint8)
    original_pixels = int(mask_bin.sum())
    if original_pixels == 0:
        continue

    for k in kernel_sizes:
        kernel = np.ones((k, k), np.uint8)
        eroded_mask = cv2.erode(mask_bin, kernel, iterations=1)
        eroded_pixels = int(eroded_mask.sum())
        lost_pixels = original_pixels - eroded_pixels
        lost_pct = 100.0 * lost_pixels / original_pixels

        loss_rows.append({
            'study_id': study_id,
            'kernel': k,
            'orig_pixels': original_pixels,
            'eroded_pixels': eroded_pixels,
            'lost_pixels': lost_pixels,
            'lost_pct': lost_pct,
        })

loss_df = pd.DataFrame(loss_rows).sort_values(['study_id', 'kernel'])
loss_df.to_csv(loss_out, index=False)

print('loss report rows:', len(loss_df))
print('saved:', loss_out.resolve())

loss report rows: 528
saved: /home/daniduhnev/projects/master-thesis/analysis/reports/mask_erosion_loss_report_PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED.csv


In [13]:
boundary_rows = []
edge_kernel = np.ones((3, 3), np.uint8)

for mask_path in mask_paths:
    study_id = mask_path.name.replace('_mask.png', '')
    image_path = masks_dir / f'{study_id}_image.png'

    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image_bgr is None or mask_gray is None:
        continue

    # ensure mask matches image orientation/shape
    image_h, image_w = image_bgr.shape[:2]
    mask_h, mask_w = mask_gray.shape[:2]
    if (image_h, image_w) != (mask_h, mask_w):
        # try simple transpose if mask was saved transposed
        if (image_h, image_w) == (mask_w, mask_h):
            mask_gray = mask_gray.T
            mask_h, mask_w = mask_gray.shape[:2]
        else:
            # fallback: resize mask to image size (nearest neighbor)
            mask_gray = cv2.resize(mask_gray, (image_w, image_h), interpolation=cv2.INTER_NEAREST)
            mask_h, mask_w = mask_gray.shape[:2]

    mask_bin = (mask_gray > 0).astype(np.uint8)

    image_pixels = int(image_h * image_w)

    for k in [0] + kernel_sizes:
        if k == 0:
            current_mask = mask_bin.copy()
        else:
            kernel = np.ones((k, k), np.uint8)
            current_mask = cv2.erode(mask_bin, kernel, iterations=1)

        inner = cv2.erode(current_mask, edge_kernel, iterations=1)
        boundary = (current_mask - inner).astype(np.uint8)

        white_pixels = (
            (image_bgr[:, :, 0] >= white_threshold)
            & (image_bgr[:, :, 1] >= white_threshold)
            & (image_bgr[:, :, 2] >= white_threshold)
        )

        boundary_pixels = int(boundary.sum())
        white_boundary_pixels = int((white_pixels & (boundary == 1)).sum())
        white_image_pixels = int(white_pixels.sum())

        if boundary_pixels > 0:
            white_boundary_pct = 100.0 * white_boundary_pixels / boundary_pixels
        else:
            white_boundary_pct = 0.0

        boundary_pct_of_image = 100.0 * boundary_pixels / image_pixels
        white_boundary_pct_of_image = 100.0 * white_boundary_pixels / image_pixels
        white_image_pct = 100.0 * white_image_pixels / image_pixels

        boundary_rows.append({
            'study_id': study_id,
            'kernel': k,
            'boundary_pixels': boundary_pixels,
            'white_boundary_pixels': white_boundary_pixels,
            'white_boundary_pct': white_boundary_pct,
            'image_pixels': image_pixels,
            'boundary_pct_of_image': boundary_pct_of_image,
            'white_boundary_pct_of_image': white_boundary_pct_of_image,
            'white_image_pixels': white_image_pixels,
            'white_image_pct': white_image_pct,
        })

boundary_df = pd.DataFrame(boundary_rows).sort_values(['study_id', 'kernel'])
boundary_df.to_csv(boundary_out, index=False)

print('boundary report rows:', len(boundary_df))
print('saved:', boundary_out.resolve())

boundary report rows: 670
saved: /home/daniduhnev/projects/master-thesis/analysis/reports/mask_erosion_boundary_white_report_PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED.csv


In [14]:
print('summary')
print('')

print('loss by kernel')
print(
    loss_df.groupby('kernel')['lost_pct']
    .agg(['mean', 'median', 'min', 'max'])
    .round(3)
)

print('')
print('white boundary overlap percent by kernel (within mask boundary only)')
print(
    boundary_df.groupby('kernel')['white_boundary_pct']
    .agg(['mean', 'median', 'min', 'max'])
    .round(3)
)

print('')
print('sanity check: boundary and white-boundary as percent of full image')
print(
    boundary_df.groupby('kernel')[['boundary_pct_of_image', 'white_boundary_pct_of_image', 'white_image_pct']]
    .agg(['mean', 'median', 'max'])
    .round(4)
)

print('')
print('top 5 highest loss cases at kernel 3')
print(
    loss_df[loss_df['kernel'] == 3]
    .sort_values('lost_pct', ascending=False)
    .head(5)[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']]
)

print('')
print('top 5 highest white-boundary overlap at kernel 3')
print(
    boundary_df[boundary_df['kernel'] == 3]
    .sort_values('white_boundary_pct', ascending=False)
    .head(5)[['study_id', 'white_boundary_pixels', 'boundary_pixels', 'white_boundary_pct', 'white_boundary_pct_of_image']]
)

summary

loss by kernel
          mean  median    min     max
kernel                               
1        0.000   0.000  0.000   0.000
2        2.670   2.457  1.281   8.198
3        5.323   4.899  2.558  16.299
5       10.578   9.752  5.101  32.215

white boundary overlap percent by kernel (within mask boundary only)
          mean  median  min     max
kernel                             
0       38.254  39.221  0.0  48.854
1       38.254  39.221  0.0  48.854
2       34.966  36.089  0.0  48.431
3        0.000   0.000  0.0   0.000
5        0.000   0.000  0.0   0.000

sanity check: boundary and white-boundary as percent of full image
       boundary_pct_of_image                 white_boundary_pct_of_image  \
                        mean  median     max                        mean   
kernel                                                                     
0                     0.0991  0.0977  0.2185                      0.0391   
1                     0.0991  0.0977  0.2185          

In [15]:
# flag studies with high mask loss at kernel 3
threshold_pct = 30.0

k3 = loss_df[loss_df['kernel'] == 3].copy()
flagged = k3[k3['lost_pct'] > threshold_pct].sort_values('lost_pct', ascending=False)

print('kernel 3 studies:', len(k3))
print('threshold (%):', threshold_pct)
print('flagged studies:', len(flagged))
print('flagged percent:', round(100.0 * len(flagged) / len(k3), 2), '%')

print('')
print('kernel 3 lost_pct quantiles')
print(k3['lost_pct'].quantile([0.75, 0.80, 0.85, 0.90, 0.95]).round(3))

print('')
print('flagged studies list')
print(flagged[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']])

flag_out = reports_dir / 'mask_erosion_loss_flags_kernel3_PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED.csv'
flagged[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']].to_csv(flag_out, index=False)
print('')
print('saved:', flag_out.resolve())

kernel 3 studies: 132
threshold (%): 30.0
flagged studies: 0
flagged percent: 0.0 %

kernel 3 lost_pct quantiles
0.75    6.039
0.80    6.415
0.85    6.640
0.90    7.786
0.95    9.370
Name: lost_pct, dtype: float64

flagged studies list
Empty DataFrame
Columns: [study_id, orig_pixels, lost_pixels, lost_pct]
Index: []

saved: /home/daniduhnev/projects/master-thesis/analysis/reports/mask_erosion_loss_flags_kernel3_PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED.csv


In [16]:
# Quick verification of where key numbers come from

# 1) Image size used in this dataset
print('image_pixels unique values:', boundary_df['image_pixels'].nunique())
print('image_pixels value:', int(boundary_df['image_pixels'].iloc[0]))

# 2) Kernel 0 averages (same logic discussed above)
k0 = boundary_df[boundary_df['kernel'] == 0]
mean_boundary_pixels = k0['boundary_pixels'].mean()
mean_white_boundary_pixels = k0['white_boundary_pixels'].mean()
mean_white_boundary_pct = k0['white_boundary_pct'].mean()
mean_white_boundary_pct_of_image = k0['white_boundary_pct_of_image'].mean()

print('')
print('kernel 0 mean boundary_pixels:', round(mean_boundary_pixels, 3))
print('kernel 0 mean white_boundary_pixels:', round(mean_white_boundary_pixels, 3))
print('kernel 0 mean white_boundary_pct (within boundary):', round(mean_white_boundary_pct, 3))
print('kernel 0 mean white_boundary_pct_of_image:', round(mean_white_boundary_pct_of_image, 4), '%')

# 3) Show one concrete study calculation
example = k0.iloc[0]
print('')
print('example study:', example['study_id'])
print('boundary_pixels:', int(example['boundary_pixels']))
print('white_boundary_pixels:', int(example['white_boundary_pixels']))
print(
    'white_boundary_pct = 100 * white_boundary_pixels / boundary_pixels =',
    round(100.0 * int(example['white_boundary_pixels']) / int(example['boundary_pixels']), 6)
)
print(
    'white_boundary_pct_of_image = 100 * white_boundary_pixels / image_pixels =',
    round(100.0 * int(example['white_boundary_pixels']) / int(example['image_pixels']), 6)
)

image_pixels unique values: 1
image_pixels value: 786432

kernel 0 mean boundary_pixels: 779.358
kernel 0 mean white_boundary_pixels: 307.231
kernel 0 mean white_boundary_pct (within boundary): 38.254
kernel 0 mean white_boundary_pct_of_image: 0.0391 %

example study: 01_01
boundary_pixels: 1002
white_boundary_pixels: 343
white_boundary_pct = 100 * white_boundary_pixels / boundary_pixels = 34.231537
white_boundary_pct_of_image = 100 * white_boundary_pixels / image_pixels = 0.043615


In [17]:
# Diagnostic: check image / mask shape consistency
mismatches = []
for mask_path in sorted(masks_dir.glob('*_mask.png')):
    study_id = mask_path.name.replace('_mask.png', '')
    image_path = masks_dir / f'{study_id}_image.png'
    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image_bgr is None or mask_gray is None:
        print('missing files for', study_id, 'image_missing=', image_bgr is None, 'mask_missing=', mask_gray is None)
        continue
    h_i,w_i = image_bgr.shape[:2]
    h_m,w_m = mask_gray.shape[:2]
    if (h_i,w_i) != (h_m,w_m):
        mismatches.append((study_id, (h_i,w_i), (h_m,w_m)))

if not mismatches:
    print('No mismatches found')
else:
    print('Mismatches (study, image_shape, mask_shape):')
    for m in mismatches:
        print(m)
    print('Total mismatches:', len(mismatches))


Mismatches (study, image_shape, mask_shape):
('03_01', (768, 1024), (1024, 768))
Total mismatches: 1
